<a href="https://colab.research.google.com/github/Seripro/c-learning/blob/main/gpu_learning/numerical_pi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sun May 24 01:30:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%%writefile numerical_pi.cu
#include <stdio.h>
#include <stdlib.h>

// ==========================================
// ★ ミッション1：GPUカーネルを完成させよ！
// ==========================================
__global__ void integrate_pi(long long N, double dx, double *d_total_area) {
    // 1. 自分の「出席番号（通し番号 i）」を計算する（モンテカルロと同じ論理だ！）
    // 【ここを君が書く！】
    long long i = blockDim.x * blockIdx.x + threadIdx.x;


    // 2. 自分の番号 i が N より小さい時だけ計算する（安全装置）
    if (i < N) {
        // 3. 自分の担当する長方形の x 座標を計算
        // 【ここを君が書く！】
        double x_coord = (i + 0.5) * dx;

        // 4. 長方形の高さを計算 ( 4.0 / (1.0 + x * x) )
        // 【ここを君が書く！】
        double y_val = 4.0 / (1.0 + x_coord * x_coord);

        // 5. 面積 (高さ * dx) を計算し、atomicAdd で d_total_area に足し込む！
        // ※CUDAの atomicAdd は double 型もそのまま使えるぞ！
        // 【ここを君が書く！】
        double area_segment = y_val * dx;
        atomicAdd(d_total_area, area_segment);
    }
}

int main(void) {
    // データの分割数（1億個の長方形に切り刻む！）
    long long N = 100000000;
    double dx = 1.0 / (double)N;

    // 軍隊の編成
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    printf("%lld 個の長方形の面積をGPUで一斉計算します...\n", N);

    // 本社と工場のメモリ準備
    double h_total_area = 0.0;
    double *d_total_area;

    cudaMalloc((void**)&d_total_area, sizeof(double));
    cudaMemcpy(d_total_area, &h_total_area, sizeof(double), cudaMemcpyHostToDevice);

    // ==========================================
    // ★ ミッション2：GPUへの出撃命令を書け！
    // ==========================================
    // integrate_pi 関数に対して、メガホン <<< >>> を使って出撃命令を出してくれ。
    // 渡す引数は N, dx, d_total_area の3つだ。
    // 【ここを君が書く！】
    integrate_pi<<<blocksPerGrid, threadsPerBlock>>>(N, dx, d_total_area);

    // 結果の回収
    cudaMemcpy(&h_total_area, d_total_area, sizeof(double), cudaMemcpyDeviceToHost);

    // 結果発表
    printf("GPUが弾き出した円周率: %.10f\n", h_total_area);

    cudaFree(d_total_area);

    return 0;
}

Overwriting numerical_pi.cu


In [2]:
!wget -q https://developer.download.nvidia.com/hpc-sdk/25.1/nvhpc_2025_251_Linux_x86_64_cuda_multi.tar.gz
!tar -xzf nvhpc_2025_251_Linux_x86_64_cuda_multi.tar.gz

In [7]:
!nvcc numerical_pi.cu -o numerical_pi -arch=sm_75

In [8]:
!./numerical_pi

100000000 個の長方形の面積をGPUで一斉計算します...
GPUが弾き出した円周率: 3.1415926536
